# Mt. Hood Ash Dispersal — Tephra2 Sweep Analysis

Post-processing, visualization, and export for the 1,000,000-run Tephra2 parameter sweep produced by
`tephra2_sweep.ipynb` (`tephra2_aggregated.csv`).

### Hazard thresholds

Ash loading thresholds below are from Wilson et al. (2014) and Jenkins et al. (2015):

| Threshold (kg/m²) | Impact |
|---:|---|
| 1 | Transport and agriculture disruption |
| 10 | Crop damage, infrastructure disruption |
| 100 | Roof collapse risk |
| 1,000 | Severe structural damage |

These thresholds are load-based, not volcano-type specific, and assume a bulk ash density of ~1,000 kg/m³.
Mt. Hood's fine lithic ash may have a lower bulk density (~500–800 kg/m³), so a given kg/m² loading corresponds
to a *thicker* deposit here than at a volcano producing denser ash — treat the mapped kg/m² values as the
physically robust quantity, and convert to a deposit thickness only with an explicit density assumption in mind.

### References
Wilson, T.M. et al. (2014); Jenkins, S.F. et al. (2015); Scott et al. (2025); Buckland et al. (2022);
Mannen (2020); Biass et al. (2016); Pardini et al. (2016); Kawamoto & Ui (2021).

## Section 1 — Setup & Data Loading

`tephra2_aggregated.csv` from a full 1,000,000-run sweep is far too large to load into a single pandas
DataFrame (on the order of tens of billions of rows) -- doing so is what was killing the kernel. Instead, this
notebook queries the CSV out-of-core with [DuckDB](https://duckdb.org/), which never needs to hold the full
table in memory.

The one-time setup below also **materializes the CSV into a local DuckDB database file**
(`tephra2_aggregated.duckdb`) rather than querying the raw CSV directly on every cell. Re-parsing a
multi-billion-row CSV from scratch for every section's query would be extremely slow; converting it once into
DuckDB's columnar on-disk format makes every later query in this notebook fast. This conversion cell is safe to
re-run (`CREATE TABLE IF NOT EXISTS` skips the work if it's already done) -- but if you re-run the sweep and
want to refresh the analysis, delete `tephra2_aggregated.duckdb` first so it gets rebuilt from the new CSV.

In [ ]:
import subprocess
subprocess.run('pip install --quiet duckdb', shell=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.interpolate import griddata
import cartopy.crs as ccrs
import rioxarray  # noqa: F401 -- registers the .rio accessor used for GeoTIFF export
import xarray as xr
import utm
import duckdb


In [ ]:
HAZARD_THRESHOLDS = [1, 10, 100, 1000]  # kg/m^2 -- Wilson et al. (2014), Jenkins et al. (2015)

# Vent location (must match tephra2_sweep.ipynb)
vent_latitude, vent_longitude = 45.22, -121.44
vent_easting, vent_northing, utm_zone_number, utm_zone_letter = utm.from_latlon(vent_latitude, vent_longitude)

vol_easting, vol_northing = round(vent_easting), round(vent_northing)
grid_spacing = 1000  # meters -- must match tephra2_sweep.ipynb

# Points of interest (must match tephra2_sweep.ipynb)
poi_names = ["Rhododendron", "Parkdale", "Govt. Camp"]
poi_locations = {
    "Rhododendron": (45.329563, -121.911191),
    "Parkdale": (45.519839, -121.596742),
    "Govt. Camp": (45.1808, -121.4509),
}


In [ ]:
CSV_PATH = "tephra2_aggregated.csv"
DB_PATH = "tephra2_aggregated.duckdb"

con = duckdb.connect(DB_PATH)
con.execute(f"""
    CREATE TABLE IF NOT EXISTS sweep AS
    SELECT
        plume_height,
        eruption_mass,
        diffusion_coef,
        easting,
        northing,
        -- Tephra2 can underflow to values below double precision's usable range; treat those as exact zero
        CASE WHEN mass_kg_m2 < 1e-300 THEN 0.0 ELSE mass_kg_m2 END AS mass_kg_m2
    FROM read_csv_auto('{CSV_PATH}')
""")

n_rows = con.execute("SELECT COUNT(*) FROM sweep").fetchone()[0]
print(f"{n_rows:,} rows in {DB_PATH}")
con.execute("SELECT * FROM sweep LIMIT 5").fetchdf()


In [ ]:
# tephra2_sweep.ipynb does not write latitude/longitude -- every run shares the same grid points,
# so it's far cheaper to convert UTM -> lat/lon once here (over the unique locations, a small result) than
# for every row.
unique_locations = con.execute("SELECT DISTINCT easting, northing FROM sweep").fetchdf()
lat, lon = utm.to_latlon(unique_locations['easting'].values, unique_locations['northing'].values,
                          utm_zone_number, utm_zone_letter)
unique_locations = unique_locations.assign(latitude=lat, longitude=lon)

print(f"{len(unique_locations):,} unique grid locations")


In [ ]:
def is_on_regular_grid(dataframe):
    """Mask selecting the regular 1 km sweep grid, excluding the 3 irregularly-placed POI points."""
    return (((dataframe['easting'] - vol_easting) % grid_spacing == 0) &
            ((dataframe['northing'] - vol_northing) % grid_spacing == 0))


# SQL equivalent of is_on_regular_grid(), for filtering directly in queries against `sweep`
REGULAR_GRID_SQL = "(easting - ?) % ? = 0 AND (northing - ?) % ? = 0"
REGULAR_GRID_PARAMS = [vol_easting, grid_spacing, vol_northing, grid_spacing]


In [ ]:
def cell_edges(centers):
    """Quadrilateral edges for pcolormesh(shading='flat') that stay pinned to the data range.

    pcolormesh(shading='auto') treats the given coordinates as cell centers and extrapolates the
    outer edges by half the center-to-center spacing -- fine with 100 finely-spaced sweep steps,
    but with only a few steps (e.g. a quick smoke-test sweep) that extrapolation is huge and pushes
    the axis well past the actual swept range. Clamping the outermost edges to the outermost centers
    keeps the axis exactly at [min(centers), max(centers)] regardless of spacing.
    """
    centers = np.asarray(centers, dtype=float)
    if len(centers) == 1:
        return np.array([centers[0] - 0.5, centers[0] + 0.5])
    midpoints = (centers[:-1] + centers[1:]) / 2
    return np.concatenate(([centers[0]], midpoints, [centers[-1]]))


In [ ]:
def nearest_grid_point(latitude, longitude):
    """Return every sweep run's row at the grid point nearest to (latitude, longitude)."""
    easting, northing, _, _ = utm.from_latlon(latitude, longitude)
    dist2 = (unique_locations['easting'] - easting) ** 2 + (unique_locations['northing'] - northing) ** 2
    nearest = unique_locations.loc[dist2.idxmin()]
    return con.execute(
        "SELECT * FROM sweep WHERE easting = ? AND northing = ?",
        [nearest['easting'], nearest['northing']],
    ).fetchdf()


def poi_dataframe(name):
    lat, lon = poi_locations[name]
    return nearest_grid_point(lat, lon)


## Section 2 — Configuration

Toggle which analyses run below.

In [ ]:
run_plume_envelope_plots = True
run_max_thickness_map    = True
run_exceedance_maps      = True
run_heatmaps             = True
run_global_max           = True
run_per_location_max     = True
run_thickness_query      = True
run_export_geotiff       = True


## Section 3 — Plume Height vs. Ash Thickness Envelopes

For each POI, at every swept plume height, this plots the min/max ash thickness envelope across all
combinations of eruption mass and diffusion coefficient. This shows how much a location's exposure is
driven by plume height alone versus the other two parameters.

Each POI's run count is small enough (one row per sweep combination -- 1,000,000 at full sweep scale) to
query in full and analyze with pandas; it's only the *entire* aggregated table across all ~40,000 grid
locations that's too large to load at once.

In [ ]:
def plot_envelope(name):
    loc_df = poi_dataframe(name)
    envelope = loc_df.groupby('plume_height')['mass_kg_m2'].agg(['min', 'max', 'median'])

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.fill_between(envelope.index, envelope['min'] + 1e-6, envelope['max'],
                     color='tab:orange', alpha=0.3, label='min-max envelope')
    ax.plot(envelope.index, envelope['median'], color='tab:orange', lw=1.5, label='median')

    for threshold in HAZARD_THRESHOLDS:
        ax.axhline(threshold, color='gray', ls=':', lw=0.8)
        ax.text(envelope.index.max(), threshold, f' {threshold} kg/m²', va='bottom', ha='right',
                fontsize=7, color='gray')

    ax.set_yscale('log')
    ax.set_xlabel('Plume height (m asl)')
    ax.set_ylabel('Ash thickness (kg/m²)')
    ax.set_title(f'{name} — ash thickness vs. plume height\n(envelope across eruption mass & diffusion coefficient)')
    ax.legend()
    fig.tight_layout()

    slug = name.lower().replace('.', '').replace(' ', '_')
    fig.savefig(f'envelope_{slug}.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
if run_plume_envelope_plots:
    plot_envelope("Rhododendron")


In [ ]:
if run_plume_envelope_plots:
    plot_envelope("Parkdale")


In [ ]:
if run_plume_envelope_plots:
    plot_envelope("Govt. Camp")


## Section 4 — Maps

Shared mapping helper: interpolates scattered (lon, lat, value) points onto a regular lon/lat mesh
(`scipy.griddata`) and draws a filled contour map with hazard-threshold contour lines, POI markers, and a
vent marker on a `cartopy.PlateCarree` projection.

In [ ]:
def hazard_map(ax, lon, lat, values, cbar_label, log_norm=True, cmap='YlOrRd',
                solid_levels=None, dashed_levels=None, poi_values=None):
    grid_lon = np.linspace(lon.min(), lon.max(), 300)
    grid_lat = np.linspace(lat.min(), lat.max(), 300)
    mesh_lon, mesh_lat = np.meshgrid(grid_lon, grid_lat)

    grid_values = griddata((lon, lat), values, (mesh_lon, mesh_lat), method='linear')

    positive_values = values[values > 0]
    if log_norm and len(positive_values) > 0:
        norm = LogNorm(vmin=max(positive_values.min(), 1e-3), vmax=values.max())
    else:
        norm = None
    cf = ax.contourf(mesh_lon, mesh_lat, grid_values, levels=50, cmap=cmap, norm=norm,
                      transform=ccrs.PlateCarree())
    cbar = plt.colorbar(cf, ax=ax, shrink=0.8, pad=0.02)
    cbar.set_label(cbar_label)

    if solid_levels:
        finite_values = grid_values[~np.isnan(grid_values)]
        gvmin, gvmax = (finite_values.min(), finite_values.max()) if len(finite_values) else (0, 0)
        solid_levels = [lvl for lvl in solid_levels if gvmax >= lvl >= gvmin]
        if solid_levels:
            cs = ax.contour(mesh_lon, mesh_lat, grid_values, levels=solid_levels, colors='k',
                             linewidths=1, transform=ccrs.PlateCarree())
            ax.clabel(cs, fmt='%g', fontsize=7)

    if dashed_levels:
        cs2 = ax.contour(mesh_lon, mesh_lat, grid_values, levels=dashed_levels, colors='k',
                          linewidths=0.5, linestyles='dashed', transform=ccrs.PlateCarree())
        ax.clabel(cs2, fmt='%g', fontsize=6)

    ax.plot(vent_longitude, vent_latitude, marker='^', color='red', markersize=10,
             transform=ccrs.PlateCarree(), zorder=5, label='Vent')

    for name, (plat, plon) in poi_locations.items():
        ax.plot(plon, plat, marker='*', color='black', markersize=10,
                 transform=ccrs.PlateCarree(), zorder=5)
        label = name if poi_values is None else f"{name}\n{poi_values.get(name, float('nan')):.3g} kg/m²"
        ax.annotate(label, (plon, plat), xytext=(5, 5), textcoords='offset points',
                    fontsize=7, transform=ccrs.PlateCarree())

    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.text(0.01, 0.01,
            'Hazard thresholds: Wilson et al. (2014), Jenkins et al. (2015)',
            transform=ax.transAxes, fontsize=6, color='gray')


### Maximum ash thickness map

Worst case per grid location across all 1,000,000 parameter combinations. The `MAX(mass_kg_m2) ... GROUP BY`
runs inside DuckDB against the full on-disk table and only the ~40,000-row per-location result comes back into
pandas.

In [ ]:
if run_max_thickness_map:
    max_by_loc = con.execute(
        f"""
        SELECT easting, northing, MAX(mass_kg_m2) AS mass_kg_m2
        FROM sweep
        WHERE {REGULAR_GRID_SQL}
        GROUP BY easting, northing
        """,
        REGULAR_GRID_PARAMS,
    ).fetchdf()
    max_by_loc = max_by_loc.merge(unique_locations, on=['easting', 'northing'], how='left')

    poi_max = {name: poi_dataframe(name)['mass_kg_m2'].max() for name in poi_names}

    fig, ax = plt.subplots(figsize=(9, 8), subplot_kw={'projection': ccrs.PlateCarree()})
    hazard_map(ax, max_by_loc['longitude'].values, max_by_loc['latitude'].values,
               max_by_loc['mass_kg_m2'].values, cbar_label='Maximum ash thickness (kg/m²)',
               solid_levels=HAZARD_THRESHOLDS, dashed_levels=[5, 50, 500], poi_values=poi_max)
    ax.set_title('Maximum ash thickness across all sweep runs')
    fig.savefig('max_ash_thickness_map.png', dpi=150, bbox_inches='tight')
    plt.show()


### Exceedance probability maps

For each hazard threshold, the fraction of sweep runs at each grid point that meet or exceed that loading --
again computed inside DuckDB, returning one small per-location result per threshold.

In [ ]:
if run_exceedance_maps:
    fig, axes = plt.subplots(2, 2, figsize=(14, 12), subplot_kw={'projection': ccrs.PlateCarree()})

    for ax, threshold in zip(axes.flat, HAZARD_THRESHOLDS):
        exceed = con.execute(
            f"""
            SELECT easting, northing, AVG(CASE WHEN mass_kg_m2 >= ? THEN 1.0 ELSE 0.0 END) AS exceed_prob
            FROM sweep
            WHERE {REGULAR_GRID_SQL}
            GROUP BY easting, northing
            """,
            [threshold] + REGULAR_GRID_PARAMS,
        ).fetchdf()
        exceed = exceed.merge(unique_locations, on=['easting', 'northing'], how='left')

        grid_lon = np.linspace(exceed['longitude'].min(), exceed['longitude'].max(), 300)
        grid_lat = np.linspace(exceed['latitude'].min(), exceed['latitude'].max(), 300)
        mesh_lon, mesh_lat = np.meshgrid(grid_lon, grid_lat)
        grid_values = griddata((exceed['longitude'], exceed['latitude']), exceed['exceed_prob'],
                                (mesh_lon, mesh_lat), method='linear')

        cf = ax.contourf(mesh_lon, mesh_lat, grid_values, levels=np.linspace(0, 1, 21), cmap='YlOrRd',
                          transform=ccrs.PlateCarree())
        plt.colorbar(cf, ax=ax, shrink=0.8, pad=0.02, label='Exceedance probability')
        ax.plot(vent_longitude, vent_latitude, marker='^', color='blue', markersize=8,
                 transform=ccrs.PlateCarree(), zorder=5)
        for name, (plat, plon) in poi_locations.items():
            ax.plot(plon, plat, marker='*', color='black', markersize=8, transform=ccrs.PlateCarree(), zorder=5)
        ax.set_title(f'P(loading ≥ {threshold} kg/m²)')
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')

    fig.suptitle('Exceedance probability across all sweep runs', y=1.0)
    fig.tight_layout()
    fig.savefig('exceedance_probability_maps.png', dpi=150, bbox_inches='tight')
    plt.show()


### Plume height vs. eruption mass heatmaps (per POI)

Ash thickness as a function of plume height (x) and eruption mass (y), averaged over diffusion coefficient, at
each POI location.

In [ ]:
if run_heatmaps:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for ax, name in zip(axes, poi_names):
        loc_df = poi_dataframe(name)
        pivot = loc_df.pivot_table(index='eruption_mass', columns='plume_height',
                                    values='mass_kg_m2', aggfunc='mean')

        positive_values = pivot.values[pivot.values > 0]
        norm = LogNorm(vmin=max(positive_values.min(), 1e-3), vmax=pivot.values.max()) if len(positive_values) else None
        x_edges = cell_edges(pivot.columns.values)
        y_edges = cell_edges(pivot.index.values)
        pcm = ax.pcolormesh(x_edges, y_edges, pivot.values,
                             norm=norm, cmap='YlOrRd', shading='flat')
        fig.colorbar(pcm, ax=ax, label='Ash thickness (kg/m²)')
        ax.set_xlabel('Plume height (m asl)')
        ax.set_ylabel('Eruption mass (kg)')
        ax.set_yscale('log')
        ax.set_title(name)

    fig.suptitle('Ash thickness vs. plume height & eruption mass (diffusion coefficient averaged)', y=1.03)
    fig.tight_layout()
    fig.savefig('plume_height_eruption_mass_heatmaps.png', dpi=150, bbox_inches='tight')
    plt.show()


## Section 5 — Maximum Thickness Analysis

### Global maximum

The single parameter combination and grid location that produced the highest ash thickness anywhere in the
sweep. `ORDER BY ... LIMIT 1` runs against the full on-disk table in DuckDB; only the single winning row is
ever pulled into pandas.

In [ ]:
if run_global_max:
    global_max_row = con.execute(
        "SELECT * FROM sweep ORDER BY mass_kg_m2 DESC LIMIT 1"
    ).fetchdf().iloc[0]
    max_lat, max_lon = utm.to_latlon(global_max_row['easting'], global_max_row['northing'],
                                      utm_zone_number, utm_zone_letter)

    print("Global maximum ash thickness:")
    print(f"  {global_max_row['mass_kg_m2']:.3g} kg/m^2")
    print(f"  plume_height   = {global_max_row['plume_height']:.0f} m")
    print(f"  eruption_mass  = {global_max_row['eruption_mass']:.3g} kg")
    print(f"  diffusion_coef = {global_max_row['diffusion_coef']:.3g} m^2/s")
    print(f"  location       = ({max_lat:.4f}, {max_lon:.4f})")


### Per-location maximum

For every grid point, the maximum ash thickness seen across all 1,000,000 runs.

In [ ]:
if run_per_location_max:
    per_location_max = con.execute(
        f"""
        SELECT easting, northing, MAX(mass_kg_m2) AS max_mass_kg_m2
        FROM sweep
        WHERE {REGULAR_GRID_SQL}
        GROUP BY easting, northing
        """,
        REGULAR_GRID_PARAMS,
    ).fetchdf()
    per_location_max = per_location_max.merge(unique_locations, on=['easting', 'northing'], how='left')
    per_location_max = per_location_max.set_index(['latitude', 'longitude'])
    print(f"{len(per_location_max):,} unique grid locations")
    per_location_max.head()


## Section 6 — Thickness Query

Given a latitude/longitude, find the nearest grid point (matched via UTM conversion against the aggregated
easting/northing columns) and return its ash thickness across all sweep runs.

In [ ]:
def query_thickness(latitude, longitude):
    """Return the aggregated sweep rows at the grid point nearest to (latitude, longitude)."""
    return nearest_grid_point(latitude, longitude)


if run_thickness_query:
    example = query_thickness(*poi_locations["Govt. Camp"])
    print(f"{len(example):,} runs at the grid point nearest Govt. Camp")
    print(example['mass_kg_m2'].describe())


## Section 7 — Export for QGIS

Exports the maximum ash thickness raster (from Section 5) as a GeoTIFF in **WGS84 / UTM Zone 10N**
(EPSG:32610), which is the correct UTM zone for Mt. Hood.

In [ ]:
UTM10N_EPSG = 32610

def build_max_thickness_grid():
    """Grid the per-location maximum ash thickness onto a regular easting/northing raster.

    Restricted to points on the regular 1 km sweep grid -- the 3 POI points are not on-grid and
    would otherwise corrupt the raster's shape and cell size.
    """
    max_by_loc = con.execute(
        f"""
        SELECT easting, northing, MAX(mass_kg_m2) AS mass_kg_m2
        FROM sweep
        WHERE {REGULAR_GRID_SQL}
        GROUP BY easting, northing
        """,
        REGULAR_GRID_PARAMS,
    ).fetchdf()

    eastings = np.sort(max_by_loc['easting'].unique())
    northings = np.sort(max_by_loc['northing'].unique())

    grid = max_by_loc.pivot(index='northing', columns='easting', values='mass_kg_m2')
    grid = grid.reindex(index=northings, columns=eastings)
    return grid, eastings, northings


In [ ]:
if run_export_geotiff:
    grid, eastings, northings = build_max_thickness_grid()

    raster = xr.DataArray(
        grid.values[::-1, :],  # north-up
        dims=("y", "x"),
        coords={"y": northings[::-1], "x": eastings},
        name="max_ash_thickness_kg_m2",
    )
    raster.rio.write_crs(f"EPSG:{UTM10N_EPSG}", inplace=True)
    raster.rio.write_nodata(np.nan, inplace=True)
    raster.rio.to_raster("max_ash_thickness.tif")
    print("Wrote max_ash_thickness.tif")
